# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

### Setup

In [ ]:
import os
import warnings
from dotenv import load_dotenv

from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_core.tools import Tool
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_classic import hub

import tiktoken

warnings.filterwarnings("ignore")
load_dotenv()

# ---- LLM ----
llm = ChatOpenAI(model="gpt-4", temperature=0)

# ---- Tavily Tool ----
tavily = TavilySearchResults()
encoding = tiktoken.encoding_for_model("gpt-4")

def safe_search(query: str) -> str:
    result = tavily.run(query)
    text = str(result)

    tokens = encoding.encode(text)
    return encoding.decode(tokens[:3500])

tools = [
    Tool(
        name="TavilySafeSearch",
        func=safe_search,
        description="Search the web safely and return trimmed results"
    )
]

# ---- ReAct Prompt ----

prompt = hub.pull("hwchase17/react")

# ---- Create Agent ----
agent = create_react_agent(llm, tools, prompt)

# ---- Executor ----
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

# ---- Run ----
agent_executor.invoke({"input": "Who is the CEO of OpenAI?"})


c:\Users\VORPC\anaconda3\envs\env1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm




> Entering new AgentExecutor chain...
I need to find out who the current CEO of OpenAI is.
Action: TavilySafeSearch
Action Input: "CEO of OpenAI"[{'title': 'OpenAI - Wikipedia', 'url': 'https://en.wikipedia.org/wiki/OpenAI', 'content': 'In February 2025, OpenAI CEO Sam Altman stated that the company is interested in collaborating with the People\'s Republic of China, despite regulatory restrictions imposed by the U.S. government. This shift comes in response to the growing influence of the Chinese artificial intelligence company DeepSeek, which has disrupted the AI market with open models, including DeepSeek V3 and DeepSeek R1. Following DeepSeek\'s market emergence, OpenAI enhanced security protocols to protect proprietary development techniques from industrial espionage. Some industry observers noted similarities between DeepSeek\'s model distillation approach and OpenAI\'s methodology, though no formal intellectual property claim was filed. [...] CEO and co-founder: Sam Altman, fo

{'input': 'Who is the CEO of OpenAI?',
 'output': 'Sam Altman is the CEO of OpenAI.'}

### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

In [ ]:
prompt_1 = """Retrieve the most recent and credible advancements in generative AI applied to healthcare during the year 2025.
Summarize the developments in a structured response in Markdown that includes:
1. Key breakthroughs or innovations with short descriptions.
2. Concrete real-world deployments, pilots, or market growth figures.
3. Examples of research papers, technical progress, or standards/guidelines.
4. Ethical, regulatory, or clinical safety considerations.
5. Major trends shaping 2026 outlook.
Organize the output clearly under headings like:
# Breakthroughs
# Real-World Deployments
# Research & Standards
# Challenges & Ethics
# Trends to Watch
"""

response_1 = agent_executor.invoke({"input": prompt_1})
print(response_1)




> Entering new AgentExecutor chain...
I need to search for the most recent and credible advancements in generative AI applied to healthcare during the year 2025. I will use TavilySafeSearch to find this information.
Action: TavilySafeSearch
Action Input: "Generative AI in healthcare advancements 2025" [{'title': '7 ways AI is transforming healthcare - The World Economic Forum', 'url': 'https://www.weforum.org/stories/2025/08/ai-transforming-global-health/', 'content': '*   [Quantcast 5](https://www.weforum.org/stories/2025/08/ai-transforming-global-health/#)[Learn more about this provider![Image 31](blob:http://localhost/7c37f61654102a88a9be209f9569ef78)](https://www.quantcast.com/privacy/ "Quantcast\'s privacy policy")**__qca**Collects data on the user\'s visits to the website, such as the number of visits, average time spent on the website and what pages have been loaded with the purpose of generating reports for optimising the website content.**Maximum Storage Duration**: 1 year**

In [ ]:
from IPython.display import display, Markdown

# Get the plain text output
text_output = response_1['output']  # <- most executors store the text here

# Display as Markdown
display(Markdown(text_output))



# Breakthroughs
- Generative Adversarial Networks (GANs), Variational Autoencoders (VAEs), Recurrent Neural Networks (RNNs), and Deep Reinforcement Learning (DRL) have been used for synthetic data generation to enhance medical datasets, improve disease diagnosis (e.g., breast cancer, vertebral fractures), enhance radiology imaging (x-ray, CT, MRI, PET imaging), and assist in medical education and drug discovery. 

# Real-World Deployments
- AI has been used to reduce physician burnout through ambient clinical documentation and recover revenue lost to coding errors and denials through coding and billing automation. Patient engagement and prior authorization have seen significant growth.
- AI tools have been used to identify signs of diabetic retinopathy during eye examinations, in robotic surgery, and for better monitoring of patient health in pediatric emergency medicine.

# Research & Standards
- International guidelines are available for the responsible use of Generative AI in clinical practice and research. 

# Challenges & Ethics
- Challenges include data quality and quantity, lack of interpretability, ethical and regulatory concerns, generalization issues, bias and fairness, and safety and reliability. 

# Trends to Watch
- The trend towards intelligent use of Generative AI in general practice is accelerating. Future applications include AI tools for automatic documentation and clinical decision support, and digital skills to use AI tools effectively.

### Exercise 2: AI Startups Landscape

Goal: Track 2025’s top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.